<a href="https://colab.research.google.com/github/AbdoNode/TeleDrive/blob/main/Copy_of_TeleDrive_PRO_fixed_v9_Fix_Download_Stop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TeleDrive + Telegram Bot — Colab (Hardened)

Control TeleDrive from Telegram after starting the Colab runtime.

**Legal use only:** use torrents and files that you have the right to download or upload.

Commands:
`/start` `/help` `/ping` `/download` `/status` `/upload` `/clear` `/stop` `/shutdown`

**Important:** the Colab runtime must remain open and running. This notebook does not start a runtime remotely.


## Telegram Secret Setup

In Colab, open **Secrets 🔑** and add:

- `TELEGRAM_BOT_TOKEN` — your bot token from BotFather.
- `TELEGRAM_ALLOWED_USER_ID` — optional; your Telegram numeric user ID to restrict access.

**Never put your bot token directly in notebook cells or share it in messages.**


In [ ]:
from google.colab import userdata

def get_secret(name, required=False):
    try:
        value = userdata.get(name)
    except Exception:
        value = None

    value = (value or "").strip()

    if required and not value:
        raise RuntimeError(
            f"Missing Colab Secret: {name}"
        )

    return value


TELEGRAM_BOT_TOKEN = get_secret(
    "TELEGRAM_BOT_TOKEN",
    required=True
)

TELEGRAM_ALLOWED_USER_ID = get_secret(
    "TELEGRAM_ALLOWED_USER_ID"
)

print("✅ Telegram credentials loaded from Colab Secrets")

if TELEGRAM_ALLOWED_USER_ID:
    print("🔐 Telegram access restriction: ENABLED")
else:
    print("⚠️ Telegram access restriction: DISABLED")
    print("   Add TELEGRAM_ALLOWED_USER_ID for private use.")


In [ ]:
import os
import subprocess

REPO_DIR = os.path.abspath("TeleDrive")
REPO_URL = "https://github.com/AbdoNode/TeleDrive.git"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("📦 TeleDrive repository already exists.")

    result = subprocess.run(
        ["git", "-C", REPO_DIR, "pull", "--ff-only"],
        text=True,
        capture_output=True
    )

    if result.returncode == 0:
        print("✅ TeleDrive repository updated.")
    else:
        print("⚠️ Could not fast-forward the existing repository.")
        print(result.stderr.strip() or result.stdout.strip())
        print("   Keeping the existing working copy.")
else:
    if os.path.exists(REPO_DIR):
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git repository. "
            "Rename/remove it and rerun this cell."
        )

    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

    print("✅ TeleDrive repository cloned.")

os.chdir(REPO_DIR)
print("📁 Working directory:", os.getcwd())


In [ ]:
import subprocess
import sys
from pathlib import Path

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        "requirements.txt",
    ],
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(
        "Dependency installation failed."
    )

# ------------------------------------------------------------
# Install the corrected English TeleDrive core files.
# The Telegram controller reads TD_PROGRESS directly from
# libtorrent, instead of guessing progress from log text.
# ------------------------------------------------------------

TORRENT_DOWNLOADER_CODE = 'import libtorrent as lt\nimport time\nimport os\nimport sys\nimport json\n\nfrom config import (\n    TORRENT_SESSION_FILE,\n    TORRENT_DOWNLOAD_PATH,\n    get_logger,\n)\n\nlogger = get_logger(__name__)\n\n\ntry:\n    from google.colab import files\n    IN_COLAB = True\nexcept ImportError:\n    IN_COLAB = False\n\n\ndef save_session(session, session_file=TORRENT_SESSION_FILE):\n    """Save the libtorrent session state."""\n    try:\n        with open(session_file, "wb") as f:\n            session_state = session.save_state()\n            f.write(lt.bencode(session_state))\n\n        logger.debug(f"Session saved to {session_file}")\n\n    except Exception as e:\n        logger.error(f"Failed to save session: {e}")\n\n\ndef load_session(session_file=TORRENT_SESSION_FILE):\n    """Load a previous libtorrent session if available."""\n    if os.path.exists(session_file):\n        try:\n            with open(session_file, "rb") as f:\n                session_data = f.read()\n\n            if not session_data:\n                raise ValueError("Session file is empty.")\n\n            session_state = lt.bdecode(session_data)\n\n            ses = lt.session()\n            ses.load_state(session_state)\n\n            logger.info(f"Session loaded from {session_file}")\n            return ses\n\n        except (RuntimeError, ValueError) as e:\n            logger.warning(\n                f"Could not load session ({e}). Starting a new session."\n            )\n\n            try:\n                os.remove(session_file)\n            except OSError:\n                pass\n\n    return lt.session()\n\n\ndef format_speed(value):\n    """Format a byte-per-second value."""\n    value = max(0, int(value))\n\n    if value >= 1024 * 1024:\n        return f"{value / (1024 * 1024):.2f} MB/s"\n\n    if value >= 1024:\n        return f"{value / 1024:.2f} KB/s"\n\n    return f"{value} B/s"\n\n\ndef format_size(value):\n    """Format a byte size."""\n    value = max(0, int(value))\n\n    if value >= 1024 ** 3:\n        return f"{value / (1024 ** 3):.2f} GB"\n\n    if value >= 1024 ** 2:\n        return f"{value / (1024 ** 2):.2f} MB"\n\n    if value >= 1024:\n        return f"{value / 1024:.2f} KB"\n\n    return f"{value} B"\n\n\ndef format_eta(seconds):\n    """Format ETA seconds as a short human-readable string."""\n    seconds = max(0, int(seconds))\n\n    if seconds <= 0:\n        return "Unknown"\n\n    hours, remainder = divmod(seconds, 3600)\n    minutes, seconds = divmod(remainder, 60)\n\n    if hours:\n        return f"{hours}h {minutes}m"\n\n    if minutes:\n        return f"{minutes}m {seconds}s"\n\n    return f"{seconds}s"\n\n\ndef print_progress(s):\n    """\n    Print both a machine-readable line for the Telegram controller\n    and a human-readable line for the notebook output.\n    """\n    progress = max(\n        0.0,\n        min(100.0, float(s.progress) * 100.0)\n    )\n\n    download_bps = max(0, int(s.download_rate))\n    upload_bps = max(0, int(s.upload_rate))\n    seeds = max(0, int(s.num_seeds))\n    peers = max(0, int(s.num_peers))\n\n    total = max(0, int(s.total_wanted))\n    downloaded = max(0, int(s.total_done))\n    remaining = max(0, total - downloaded)\n\n    if download_bps > 0 and remaining > 0:\n        eta_seconds = int(remaining / download_bps)\n    else:\n        eta_seconds = 0\n\n    # IMPORTANT:\n    # The controller reads this line. It intentionally contains\n    # only one percentage value, so it can never confuse a fixed\n    # "100%" denominator with the actual progress.\n    print(\n        "\\rTD_PROGRESS "\n        f"percent={progress:.2f} "\n        f"download_bps={download_bps} "\n        f"upload_bps={upload_bps} "\n        f"seeds={seeds} "\n        f"peers={peers} "\n        f"eta_seconds={eta_seconds} "\n        f"downloaded={downloaded} "\n        f"total={total}",\n        end="",\n        flush=True,\n    )\n\n    bar_length = 30\n    filled = int(bar_length * progress / 100.0)\n    filled = max(0, min(bar_length, filled))\n\n    bar = "█" * filled + "░" * (bar_length - filled)\n\n    print(\n        "\\r"\n        f"{bar} {progress:.1f}% | "\n        f"Download: {format_speed(download_bps)} | "\n        f"Upload: {format_speed(upload_bps)} | "\n        f"Seeds: {seeds} | "\n        f"Peers: {peers} | "\n        f"ETA: {format_eta(eta_seconds)}",\n        end="",\n        flush=True,\n    )\n\n    return {\n        "percent": progress,\n        "download_bps": download_bps,\n        "upload_bps": upload_bps,\n        "seeds": seeds,\n        "peers": peers,\n        "eta_seconds": eta_seconds,\n        "downloaded": downloaded,\n        "total": total,\n    }\n\n\ndef download_torrent(\n    source,\n    download_path=TORRENT_DOWNLOAD_PATH,\n    session_file=TORRENT_SESSION_FILE,\n    auto_resume=True,\n    selected_files=None,\n):\n    """\n    Download a .torrent file or Magnet link with libtorrent.\n    """\n\n    os.makedirs(download_path, exist_ok=True)\n\n    is_resuming = (\n        auto_resume\n        and os.path.exists(session_file)\n    )\n\n    ses = (\n        load_session(session_file)\n        if auto_resume\n        else lt.session()\n    )\n\n    ses.apply_settings({\n        "listen_interfaces": "0.0.0.0:6881",\n    })\n\n    params = lt.add_torrent_params()\n    params.save_path = download_path\n    params.storage_mode = (\n        lt.storage_mode_t.storage_mode_sparse\n    )\n\n    if source.startswith("magnet:"):\n        params.url = source\n        logger.info(\n            f"Adding Magnet link: {source[:60]}..."\n        )\n\n    elif source.lower().endswith(".torrent"):\n        if not os.path.exists(source):\n            logger.error(\n                f"Torrent file not found: {source}"\n            )\n            return None\n\n        try:\n            with open(source, "rb") as f:\n                torrent_data = lt.bdecode(f.read())\n\n            info = lt.torrent_info(torrent_data)\n            params.ti = info\n\n            logger.info(\n                f"Adding torrent file: {source}"\n            )\n\n        except Exception as e:\n            logger.error(\n                f"Failed to read torrent file: {e}"\n            )\n            return None\n\n    else:\n        logger.error(\n            "Invalid source. Provide a .torrent file or Magnet link."\n        )\n        return None\n\n    try:\n        handle = ses.add_torrent(params)\n\n        logger.info(\n            f"Downloading to: {download_path}"\n        )\n\n    except Exception as e:\n        logger.error(\n            f"Failed to add torrent: {e}"\n        )\n        return None\n\n    logger.info("Waiting for torrent metadata...")\n\n    while not handle.status().has_metadata:\n        time.sleep(1)\n\n    torrent_name = handle.status().name\n    file_count = 0\n\n    # Send torrent metadata once for the first Telegram message.\n    try:\n        torrent_info = handle.torrent_file()\n        torrent_files = torrent_info.files()\n        file_count = torrent_info.num_files()\n        files_list = []\n\n        for index in range(file_count):\n            try:\n                file_path = torrent_files.file_path(index)\n            except AttributeError:\n                file_path = torrent_files.at(index).path\n            files_list.append(str(file_path))\n\n        metadata = {"name": str(torrent_name), "files": files_list}\n        print(\n            "TD_META "\n            + json.dumps(metadata, ensure_ascii=False, separators=(",", ":")),\n            flush=True,\n        )\n\n    except Exception as e:\n        logger.warning(f"Could not read torrent file list: {e}")\n\n    if selected_files is not None:\n        try:\n            selected_indexes = sorted({int(i) for i in selected_files})\n            priorities = [0] * file_count\n            for file_index in selected_indexes:\n                if 1 <= file_index <= file_count:\n                    priorities[file_index - 1] = 4\n            if not any(priorities):\n                logger.error("No valid files were selected.")\n                return None\n            handle.prioritize_files(priorities)\n            logger.info(f"Selected {sum(1 for p in priorities if p > 0)} of {file_count} torrent file(s).")\n        except Exception as e:\n            logger.error(f"Failed to apply file selection: {e}")\n            return None\n\n    logger.info(\n        f"Downloading: {torrent_name}"\n    )\n\n    last_session_save = 0\n\n    try:\n        while True:\n            s = handle.status()\n\n            # Do not wait for the torrent to enter "seeding".\n            # The download is finished as soon as all wanted bytes are complete.\n            # This is especially important when only selected files are downloaded.\n            wanted_done = getattr(s, "total_wanted_done", 0)\n            wanted_total = getattr(s, "total_wanted", 0)\n            progress = getattr(s, "progress", 0.0)\n\n            download_complete = (\n                s.state == lt.torrent_status.seeding\n                or (\n                    wanted_total > 0\n                    and wanted_done >= wanted_total\n                )\n                or progress >= 0.999999\n            )\n\n            print_progress(s)\n\n            if download_complete:\n                break\n\n            now = time.time()\n\n            if now - last_session_save >= 10:\n                save_session(ses, session_file)\n                last_session_save = now\n\n            time.sleep(1)\n\n    except KeyboardInterrupt:\n        print()\n\n        logger.warning(\n            "Download interrupted. Session saved for resume."\n        )\n\n        save_session(ses, session_file)\n        return None\n\n    print()\n\n    logger.info("Download completed successfully.")\n\n    if os.path.exists(session_file):\n        try:\n            os.remove(session_file)\n            logger.debug(\n                "Session file removed after successful completion."\n            )\n        except Exception as e:\n            logger.warning(\n                f"Could not remove session file: {e}"\n            )\n\n    downloaded_path = os.path.join(\n        download_path,\n        torrent_name,\n    )\n\n    return downloaded_path\n\n\ndef get_download_status(\n    session_file=TORRENT_SESSION_FILE,\n):\n    """Return True when a resumable session exists."""\n    return os.path.exists(session_file)\n\n\ndef clear_session(\n    session_file=TORRENT_SESSION_FILE,\n):\n    """Delete the saved download session."""\n    if os.path.exists(session_file):\n        try:\n            os.remove(session_file)\n            logger.info("Download session cleared.")\n            return True\n\n        except Exception as e:\n            logger.error(\n                f"Failed to clear session: {e}"\n            )\n            return False\n\n    return True\n\n\nif __name__ == "__main__":\n    if len(sys.argv) < 2:\n        print(\n            "Usage: python torrent_downloader.py "\n            "<torrent_file/magnet_link>"\n        )\n        sys.exit(1)\n\n    source = sys.argv[1]\n    result = download_torrent(source)\n\n    if result:\n        print(f"\\nDownloaded to: {result}")\n        sys.exit(0)\n\n    sys.exit(1)\n'
MAIN_CODE = '#!/usr/bin/env python3\n\nimport sys\nimport argparse\nimport os\n\nfrom torrent_downloader import (\n    download_torrent,\n    get_download_status,\n    clear_session,\n)\nfrom config import (\n    ConfigManager,\n    TORRENT_DOWNLOAD_PATH,\n    get_logger,\n)\n\nlogger = get_logger(__name__)\n\n\ndef get_uploader():\n    """Import and return the Google Drive upload function."""\n    try:\n        from gdrive_uploader import upload_to_google_drive\n        return upload_to_google_drive\n\n    except ImportError as e:\n        logger.error(f"Failed to import uploader: {e}")\n\n        print("\\n" + "=" * 60)\n        print("ERROR: Google Drive uploader could not be imported")\n        print("=" * 60)\n        print("Install the required packages:")\n        print(\n            "  pip install google-auth-oauthlib "\n            "google-auth-httplib2 google-api-python-client"\n        )\n        print("=" * 60)\n        raise\n\n\ndef parse_arguments():\n    """Create the CLI parser and parse command-line arguments."""\n\n    parser = argparse.ArgumentParser(\n        description=(\n            "Download Torrent files and upload them to Google Drive "\n            "(optimized for Colab)."\n        ),\n        formatter_class=argparse.RawDescriptionHelpFormatter,\n        epilog="""\nExamples:\n\n  # Download a torrent file\n  python main.py download -t movie.torrent\n\n  # Use a Magnet link\n  python main.py download -t "magnet:?xt=urn:btih:..."\n\n  # Download and upload to Google Drive\n  python main.py download -t movie.torrent --upload -f FOLDER_ID\n\n  # Upload an existing file/folder\n  python main.py upload -p /path/to/folder -f FOLDER_ID\n\n  # Upload without skipping existing files\n  python main.py upload -p /path --no-skip\n\n  # Check for a resumable download\n  python main.py status\n\n  # Clear the saved download session\n  python main.py clear\n""",\n    )\n\n    subparsers = parser.add_subparsers(\n        dest="command",\n        help="Command to execute",\n    )\n\n    download_parser = subparsers.add_parser(\n        "download",\n        help="Download a Torrent",\n    )\n\n    download_parser.add_argument(\n        "-t",\n        "--torrent",\n        type=str,\n        required=True,\n        help="Path to a .torrent file or a Magnet link",\n    )\n\n    download_parser.add_argument(\n        "-d",\n        "--destination",\n        type=str,\n        default=TORRENT_DOWNLOAD_PATH,\n        help=(\n            "Download destination "\n            f"(default: {TORRENT_DOWNLOAD_PATH})"\n        ),\n    )\n\n    download_parser.add_argument(\n        "--no-resume",\n        action="store_true",\n        help="Start a new download and ignore the previous session",\n    )\n\n    download_parser.add_argument(\n        "--files",\n        type=str,\n        help="Comma-separated 1-based torrent file numbers (example: 1,2,3,...etc).",\n    )\n\n    download_parser.add_argument(\n        "--upload",\n        action="store_true",\n        help="Upload the downloaded content to Google Drive",\n    )\n\n    download_parser.add_argument(\n        "-f",\n        "--folder-id",\n        type=str,\n        help="Google Drive destination folder ID",\n    )\n\n    download_parser.add_argument(\n        "--no-skip",\n        action="store_true",\n        help="Upload files even if they already exist",\n    )\n\n    upload_parser = subparsers.add_parser(\n        "upload",\n        help="Upload files to Google Drive",\n    )\n\n    upload_parser.add_argument(\n        "-p",\n        "--path",\n        type=str,\n        required=True,\n        help="Local file or folder to upload",\n    )\n\n    upload_parser.add_argument(\n        "-f",\n        "--folder-id",\n        type=str,\n        help="Google Drive destination folder ID",\n    )\n\n    upload_parser.add_argument(\n        "--no-skip",\n        action="store_true",\n        help="Upload files even if they already exist",\n    )\n\n    subparsers.add_parser(\n        "status",\n        help="Check download session status",\n    )\n\n    subparsers.add_parser(\n        "clear",\n        help="Clear the saved download session",\n    )\n\n    return parser, parser.parse_args()\n\n\ndef handle_download(args):\n    """Download a Torrent and optionally upload it."""\n\n    print("=" * 60)\n    print("TORRENT DOWNLOADER")\n    print("=" * 60)\n\n    logger.info(f"Starting download: {args.torrent}")\n\n    selected_files = None\n    if args.files:\n        try:\n            selected_files = [int(item.strip()) for item in args.files.split(",") if item.strip()]\n        except ValueError:\n            logger.error("Invalid --files value. Use numbers like 1,2,3,...etc.")\n            return 1\n\n    downloaded_path = download_torrent(\n        args.torrent,\n        download_path=args.destination,\n        auto_resume=not args.no_resume,\n        selected_files=selected_files,\n    )\n\n    if not downloaded_path:\n        logger.error("Download failed or was cancelled.")\n        return 1\n\n    logger.info(\n        f"Download completed: {downloaded_path}"\n    )\n\n    if args.upload:\n        print("\\n" + "=" * 60)\n        print("UPLOADING TO GOOGLE DRIVE")\n        print("=" * 60)\n\n        try:\n            upload_to_google_drive = get_uploader()\n\n            results = upload_to_google_drive(\n                downloaded_path,\n                args.folder_id,\n                skip_existing=not args.no_skip,\n            )\n\n            if results["failed"]:\n                logger.warning(\n                    f"Some uploads failed "\n                    f"({len(results[\'failed\'])} items)."\n                )\n                return 1\n\n            logger.info("Upload completed successfully.")\n\n        except RuntimeError as e:\n            print("\\n" + "=" * 60)\n            print("UPLOAD ERROR")\n            print("=" * 60)\n            print(str(e))\n            print("=" * 60)\n            return 1\n\n        except Exception as e:\n            logger.error(f"Upload failed: {e}")\n            return 1\n\n    return 0\n\n\ndef handle_upload(args):\n    """Upload an existing file or folder to Google Drive."""\n\n    print("=" * 60)\n    print("GOOGLE DRIVE UPLOADER")\n    print("=" * 60)\n\n    if not os.path.exists(args.path):\n        logger.error(f"Path does not exist: {args.path}")\n        return 1\n\n    try:\n        upload_to_google_drive = get_uploader()\n\n        results = upload_to_google_drive(\n            args.path,\n            args.folder_id,\n            skip_existing=not args.no_skip,\n        )\n\n        if results["failed"]:\n            logger.warning(\n                f"Some uploads failed "\n                f"({len(results[\'failed\'])} items)."\n            )\n            return 1\n\n        logger.info("Upload completed successfully.")\n        return 0\n\n    except RuntimeError as e:\n        print("\\n" + "=" * 60)\n        print("UPLOAD ERROR")\n        print("=" * 60)\n        print(str(e))\n        print("=" * 60)\n        return 1\n\n    except Exception as e:\n        logger.error(f"Upload failed: {e}")\n        return 1\n\n\ndef handle_status(args):\n    """Check whether a resumable download session exists."""\n\n    if get_download_status():\n        print("✓ A resumable download session was found.")\n        print(\n            "  Run: python main.py download "\n            "-t <torrent>"\n        )\n    else:\n        print("✗ No resumable download session was found.")\n\n    return 0\n\n\ndef handle_clear(args):\n    """Clear the saved download session."""\n\n    if clear_session():\n        print("✓ Download session cleared.")\n        return 0\n\n    print("✗ Failed to clear the download session.")\n    return 1\n\n\ndef main():\n    """Program entry point."""\n\n    parser, args = parse_arguments()\n\n    if not args.command:\n        parser.print_help()\n        return 1\n\n    try:\n        if args.command == "download":\n            return handle_download(args)\n\n        if args.command == "upload":\n            return handle_upload(args)\n\n        if args.command == "status":\n            return handle_status(args)\n\n        if args.command == "clear":\n            return handle_clear(args)\n\n        logger.error(f"Unknown command: {args.command}")\n        return 1\n\n    except KeyboardInterrupt:\n        print("\\n\\nOperation cancelled by user.")\n\n        if args.command == "download":\n            print(\n                "Download progress was saved when possible. "\n                "Run the same command to resume."\n            )\n\n        return 130\n\n    except Exception as e:\n        logger.error(\n            f"Operation failed: {e}",\n            exc_info=True,\n        )\n        return 1\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
Path("torrent_downloader.py").write_text(
    TORRENT_DOWNLOADER_CODE,
    encoding="utf-8",
)

Path("main.py").write_text(
    MAIN_CODE,
    encoding="utf-8",
)

print("✅ TeleDrive dependencies installed")
print("✅ torrent_downloader.py updated")
print("✅ main.py updated")
print("✅ Real libtorrent progress reporting enabled")


In [ ]:
from google.colab import auth

auth.authenticate_user()

print("✅ Google Drive authentication successful")


In [ ]:
import requests

print("requests version:", requests.__version__)
print("✅ Telegram HTTP dependency ready")


## Run the Bot

Run the next cell once and leave it running.

Features:
- Only one TeleDrive task can run at a time.
- `/stop` stops the current task without affecting future tasks.
- `/shutdown` safely stops polling and the active subprocess.
- `.torrent` files sent to Telegram are downloaded to Colab and started automatically.
- Real progress comes directly from libtorrent.
- Telegram displays percentage, download speed, upload speed, seeds, peers, ETA, and downloaded/total size.
- One Telegram message is updated at a controlled interval.
- Errors are shown in Telegram instead of being hidden.


In [ ]:
# ============================================================
# TeleDrive Telegram Controller — Hardened Version
# ============================================================

import os
import sys
import re
import json
import time
import signal
import threading
import subprocess
from pathlib import Path
from urllib.parse import quote

import requests
import selectors

API = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}"

# ------------------------------------------------------------
# Global controller state
# ------------------------------------------------------------

active_task = None
active_lock = threading.RLock()

pending_selections = {}
pending_selection_lock = threading.RLock()

shutdown_event = threading.Event()
# Always start a fresh bot session, even if this cell is re-run.
shutdown_event.clear()

TORRENT_DIR = Path("incoming_torrents").resolve()
TORRENT_DIR.mkdir(parents=True, exist_ok=True)

# Telegram should not receive an edit every few milliseconds.
TELEGRAM_EDIT_INTERVAL = 1.5

# ------------------------------------------------------------
# Telegram helpers
# ------------------------------------------------------------

def tg(method, data=None, timeout=60):
    """Call Telegram Bot API and raise a useful error on failure."""
    response = requests.post(
        f"{API}/{method}",
        data=data or {},
        timeout=timeout
    )

    response.raise_for_status()

    result = response.json()

    if not result.get("ok"):
        raise RuntimeError(
            result.get(
                "description",
                f"Telegram API error in {method}"
            )
        )

    return result


def send(chat_id, text):
    """Send a Telegram message, safely splitting long text."""
    text = str(text or " ")

    for start in range(0, len(text), 3900):
        tg(
            "sendMessage",
            {
                "chat_id": chat_id,
                "text": text[start:start + 3900]
            }
        )


def create_status_message(
    chat_id,
    text="⚙️ TeleDrive task starting..."
):
    result = tg(
        "sendMessage",
        {
            "chat_id": chat_id,
            "text": text
        }
    )

    return result["result"]["message_id"]


def edit_message(chat_id, message_id, text):
    """
    Edit one Telegram message.

    Harmless 'message is not modified' errors are ignored.
    Other errors are logged but do not kill the TeleDrive process.
    """
    text = str(text or " ")

    if len(text) > 3900:
        text = text[-3900:]

    try:
        tg(
            "editMessageText",
            {
                "chat_id": chat_id,
                "message_id": message_id,
                "text": text
            }
        )

        return True

    except Exception as exc:
        message = str(exc).lower()

        if "message is not modified" in message:
            return True

        print("Telegram edit error:", repr(exc))
        return False


def allowed(uid):
    """Restrict the bot when TELEGRAM_ALLOWED_USER_ID is configured."""
    if not TELEGRAM_ALLOWED_USER_ID:
        return True

    return str(uid) == str(
        TELEGRAM_ALLOWED_USER_ID
    )


# ------------------------------------------------------------
# Process helpers
# ------------------------------------------------------------

def terminate_process(process, force=False):
    """Terminate a process group, falling back to the process itself."""
    if process is None:
        return

    try:
        if process.poll() is not None:
            return
    except Exception:
        pass

    sig = signal.SIGKILL if force else signal.SIGTERM

    try:
        os.killpg(
            os.getpgid(process.pid),
            sig
        )
        return

    except Exception:
        pass

    try:
        if force:
            process.kill()
        else:
            process.terminate()

    except Exception:
        pass


def task_running():
    with active_lock:
        task = active_task

        if task is None:
            return False

        process = task.get("process")

        # Reservation exists before process starts.
        if process is None:
            return True

        return process.poll() is None


def reserve_task(chat_id, message_id, cmd):
    """
    Atomically reserve the single global TeleDrive slot.

    Returns (task, None) on success.
    Returns (None, reason) when another task owns the slot.
    """
    global active_task

    with active_lock:

        if active_task is not None:

            old_process = active_task.get("process")

            if (
                old_process is None
                or old_process.poll() is None
            ):
                return None, "busy"

            # Clean stale finished task.
            active_task = None

        task = {
            "chat_id": chat_id,
            "message_id": message_id,
            "cmd": list(cmd),
            "process": None,
            "stop_event": threading.Event(),
            "started_at": time.time(),
        }

        active_task = task

        return task, None


def release_task(task):
    global active_task

    with active_lock:
        if active_task is task:
            active_task = None


# ------------------------------------------------------------
# Progress parser
# ------------------------------------------------------------

ANSI_RE = re.compile(
    r"\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])"
)

TD_PROGRESS_RE = re.compile(
    r"TD_PROGRESS\s+"
    r"percent=(?P<percent>\d+(?:\.\d+)?)\s+"
    r"download_bps=(?P<download>\d+)\s+"
    r"upload_bps=(?P<upload>\d+)\s+"
    r"seeds=(?P<seeds>\d+)\s+"
    r"peers=(?P<peers>\d+)\s+"
    r"eta_seconds=(?P<eta>\d+)\s+"
    r"downloaded=(?P<downloaded>\d+)\s+"
    r"total=(?P<total>\d+)"
)


def clean_output(text):
    text = ANSI_RE.sub("", text)
    return text.replace("\x00", "").strip()


def parse_metadata_line(line):
    """Parse the one-time TD_META torrent metadata line."""
    line = clean_output(line)
    if not line.startswith("TD_META "):
        return None
    try:
        payload = json.loads(line[len("TD_META "):].strip())
        files = payload.get("files") or []
        if not isinstance(files, list):
            files = []
        return {"name": str(payload.get("name") or "Unknown torrent"), "files": [str(x) for x in files]}
    except Exception:
        return None

def render_metadata(metadata):
    name = metadata.get("name", "Unknown torrent")
    files = metadata.get("files", [])
    lines = ["🚀 TeleDrive", "", f"📦 Torrent: {name}", "", "📁 Files:"]
    for index, file_name in enumerate(files, start=1):
        candidate = f"{index}. {file_name}"
        if sum(len(x) + 1 for x in lines) + len(candidate) + 120 > 3800:
            lines.append(f"... and {len(files) - index + 1} more file(s)")
            break
        lines.append(candidate)
    lines.extend(["", "📥 Select the files to download:", "/select 1,2,3,...etc", "or /select all to download everything."])
    return "\n".join(lines)


def parse_progress_line(line, progress):
    """
    Parse only the machine-readable TD_PROGRESS line.

    This deliberately ignores arbitrary percentages in logs,
    so values such as a fixed "100%" denominator can never
    become the download percentage.
    """
    line = clean_output(line)

    if not line:
        return False

    match = TD_PROGRESS_RE.search(line)

    if not match:
        return False

    data = match.groupdict()

    progress["percent"] = max(
        0.0,
        min(100.0, float(data["percent"]))
    )

    progress["download_bps"] = int(data["download"])
    progress["upload_bps"] = int(data["upload"])
    progress["seeds"] = int(data["seeds"])
    progress["peers"] = int(data["peers"])
    progress["eta_seconds"] = int(data["eta"])
    progress["downloaded"] = int(data["downloaded"])
    progress["total"] = int(data["total"])
    progress["status"] = "Downloading..."
    progress["raw"] = line

    return True


def format_bytes(value):
    value = max(0, int(value))

    if value >= 1024 ** 3:
        return f"{value / (1024 ** 3):.2f} GB"

    if value >= 1024 ** 2:
        return f"{value / (1024 ** 2):.2f} MB"

    if value >= 1024:
        return f"{value / 1024:.2f} KB"

    return f"{value} B"


def format_speed(value):
    value = max(0, int(value))

    if value >= 1024 * 1024:
        return f"{value / (1024 * 1024):.2f} MB/s"

    if value >= 1024:
        return f"{value / 1024:.2f} KB/s"

    return f"{value} B/s"


def format_eta(seconds):
    seconds = int(seconds)

    if seconds <= 0:
        return "Unknown"

    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    if hours:
        return f"{hours}h {minutes}m"

    if minutes:
        return f"{minutes}m {seconds}s"

    return f"{seconds}s"


def create_progress_bar(percent, width=50):
    """Create a single-line high-resolution Unicode progress bar."""
    try:
        percent = max(0.0, min(100.0, float(percent)))
    except (TypeError, ValueError):
        percent = 0.0

    # 8 fractional levels give ~0.25% visual resolution with 50 cells.
    fractions = ["", "▏", "▎", "▍", "▌", "▋", "▊", "▉", "█"]

    units = (percent / 100.0) * width
    full = int(units)
    remainder = units - full
    partial = int(round(remainder * 8))

    if partial >= 8:
        full += 1
        partial = 0

    full = min(full, width)
    empty = max(0, width - full - (1 if partial else 0))

    return ("█" * full) + (fractions[partial] if partial else "") + ("─" * empty)

def render_progress(progress, logs):
    """Render Telegram progress with the bar kept on ONE line."""
    percent = progress.get("percent")
    if percent is None:
        percent = 0.0

    bar = create_progress_bar(percent, width=50)

    def fmt_speed(value):
        try:
            value = float(value or 0)
        except (TypeError, ValueError):
            value = 0
        if value >= 1024 * 1024:
            return f"{value / (1024 * 1024):.2f} MB/s"
        if value >= 1024:
            return f"{value / 1024:.2f} KB/s"
        return f"{value:.0f} B/s"

    def fmt_size(value):
        try:
            value = float(value or 0)
        except (TypeError, ValueError):
            value = 0
        units = ["B", "KB", "MB", "GB", "TB"]
        idx = 0
        while value >= 1024 and idx < len(units) - 1:
            value /= 1024
            idx += 1
        return f"{value:.2f} {units[idx]}"

    def fmt_eta(seconds):
        try:
            seconds = float(seconds or 0)
        except (TypeError, ValueError):
            return "Unknown"
        if seconds <= 0:
            return "Unknown"
        seconds = int(seconds)
        if seconds < 60:
            return f"{seconds}s"
        if seconds < 3600:
            return f"{seconds // 60}m {seconds % 60}s"
        return f"{seconds // 3600}h {(seconds % 3600) // 60}m"

    seeds = int(progress.get("seeds") or 0)
    peers = int(progress.get("peers") or 0)
    downloaded = progress.get("downloaded") or 0
    total = progress.get("total") or 0

    lines = [
        "📊 Download Progress",
        "",
        f"{bar}  {float(percent):.1f}%",
        "",
        f"📥 Download: {fmt_speed(progress.get('download_bps'))}",
        f"📤 Upload: {fmt_speed(progress.get('upload_bps'))}",
        f"🌱 Seeds: {seeds}",
        f"👥 Peers: {peers}",
        f"📦 Downloaded: {fmt_size(downloaded)} / {fmt_size(total)}" if total else f"📦 Downloaded: {fmt_size(downloaded)}",
        f"⏱ ETA: {fmt_eta(progress.get('eta_seconds'))}",
        f"🔄 Status: {progress.get('status') or 'Downloading...'}",
    ]

    return "\n".join(lines)

# ------------------------------------------------------------
# Main task runner
# ------------------------------------------------------------
# ------------------------------------------------------------

def run_cmd(chat_id, cmd, message_id=None):
    """Run one TeleDrive command with separate metadata and progress messages."""
    if message_id is None:
        message_id = create_status_message(chat_id, "⚙️ TeleDrive task starting...")

    task, reason = reserve_task(chat_id, message_id, cmd)
    if task is None:
        edit_message(chat_id, message_id, "⚠️ Another TeleDrive task is already running.")
        return False

    process = None
    selector = None
    progress_message_id = None
    metadata_received = False
    progress = {"percent": None, "download_bps": 0, "upload_bps": 0, "seeds": 0, "peers": 0, "eta_seconds": 0, "downloaded": 0, "total": 0, "status": "Starting...", "raw": None}
    logs = []
    buffer = ""
    last_edit = 0
    last_text = None

    def current_message_id():
        return progress_message_id or message_id

    def handle_piece(piece):
        nonlocal metadata_received, progress_message_id
        piece = clean_output(piece)
        if not piece:
            return
        metadata = parse_metadata_line(piece)
        if metadata is not None and not metadata_received:
            metadata_received = True
            edit_message(chat_id, message_id, render_metadata(metadata))
            progress_message_id = create_status_message(chat_id, "📊 Download Progress\n\n⏳ Waiting for torrent data...")
            task["message_id"] = progress_message_id
            return
        if parse_progress_line(piece, progress):
            return
        logs.append(piece)
        del logs[:-8]

    try:
        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"
        command = list(cmd)
        if command and command[0] in ("python", "python3"):
            command[0] = sys.executable
        if len(command) >= 2 and command[0] == sys.executable and command[1] == "main.py":
            command.insert(1, "-u")

        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, bufsize=0, env=env, start_new_session=True)
        with active_lock:
            if active_task is not task:
                terminate_process(process)
                return False
            task["process"] = process

        print("▶️ Running:", " ".join(repr(str(x)) for x in command))
        selector = selectors.DefaultSelector()
        if process.stdout is not None:
            selector.register(process.stdout, selectors.EVENT_READ)
        decoder = __import__("codecs").getincrementaldecoder("utf-8")(errors="replace")

        def process_chunk(chunk):
            nonlocal buffer
            if not chunk:
                return
            buffer += decoder.decode(chunk, final=False)
            buffer = buffer.replace("\r\n", "\n").replace("\r", "\n")
            pieces = buffer.split("\n")
            buffer = pieces[-1]
            for piece in pieces[:-1]:
                handle_piece(piece)

        while True:
            if task["stop_event"].is_set():
                break
            if shutdown_event.is_set():
                task["stop_event"].set()
                break
            events = selector.select(timeout=0.5)
            for key, _ in events:
                try:
                    chunk = os.read(key.fileobj.fileno(), 8192)
                except OSError:
                    chunk = b""
                if chunk:
                    process_chunk(chunk)
            now = time.time()
            if progress_message_id is not None and now - last_edit >= TELEGRAM_EDIT_INTERVAL:
                current_text = render_progress(progress, logs)
                if current_text != last_text:
                    edit_message(chat_id, progress_message_id, current_text)
                    last_text = current_text
                    last_edit = now
            if process.poll() is not None:
                break

        stopped = task["stop_event"].is_set() or shutdown_event.is_set()
        if stopped and process.poll() is None:
            edit_message(chat_id, current_message_id(), "🛑 Stopping TeleDrive...")
            terminate_process(process, force=False)
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                terminate_process(process, force=True)
                try:
                    process.wait(timeout=3)
                except subprocess.TimeoutExpired:
                    pass

        if process.stdout is not None:
            try:
                while True:
                    chunk = os.read(process.stdout.fileno(), 8192)
                    if not chunk:
                        break
                    process_chunk(chunk)
            except Exception:
                pass
        try:
            buffer += decoder.decode(b"", final=True)
        except Exception:
            pass
        remaining = clean_output(buffer)
        if remaining:
            handle_piece(remaining)

        return_code = process.wait()
        if stopped:
            final_title = "🛑 TeleDrive stopped."
        elif return_code == 0:
            final_title = "✅ Download completed."
            if progress["percent"] is not None:
                progress["percent"] = 100.0
            progress["status"] = "Completed."
        else:
            final_title = "❌ TeleDrive failed."

        final_id = current_message_id()
        final_text = render_progress(progress, logs)
        final_lines = final_text.splitlines()
        if final_lines:
            final_lines[0] = final_title
        final_text = "\n".join(final_lines) + f"\n\nExit code: {return_code}"
        edit_message(chat_id, final_id, final_text)
        print(f"⏹ TeleDrive exit code: {return_code}")
        return return_code == 0

    except Exception as exc:
        print("❌ TeleDrive task exception:", repr(exc))
        if process is not None:
            terminate_process(process, force=True)
        edit_message(chat_id, current_message_id(), "❌ TeleDrive task error:\n\n" + str(exc))
        return False

    finally:
        if selector is not None:
            try:
                selector.close()
            except Exception:
                pass
        release_task(task)


# ------------------------------------------------------------
# Telegram .torrent handling
# ------------------------------------------------------------

def safe_torrent_filename(filename):
    """Create a local filename without path traversal."""
    filename = os.path.basename(
        filename or "upload.torrent"
    )

    # Remove control characters.
    filename = re.sub(
        r"[\x00-\x1f\x7f]",
        "_",
        filename
    )

    if not filename.lower().endswith(".torrent"):
        filename += ".torrent"

    if filename in (
        "",
        ".torrent",
    ):
        filename = "upload.torrent"

    return filename


def download_telegram_torrent(file_id, filename):
    """Download a Telegram document to incoming_torrents."""
    info = tg(
        "getFile",
        {
            "file_id": file_id
        }
    )

    telegram_path = info["result"]["file_path"]

    filename = safe_torrent_filename(
        filename
    )

    destination = (
        TORRENT_DIR / filename
    )

    # Never overwrite an existing torrent.
    stem = destination.stem
    suffix = destination.suffix
    counter = 1

    while destination.exists():

        destination = (
            TORRENT_DIR
            / f"{stem}_{counter}{suffix}"
        )

        counter += 1

    url = (
        "https://api.telegram.org/file/"
        f"bot{TELEGRAM_BOT_TOKEN}/"
        f"{quote(telegram_path)}"
    )

    temp = destination.with_suffix(
        destination.suffix + ".part"
    )

    try:

        with requests.get(
            url,
            stream=True,
            timeout=(15, 120)
        ) as response:

            response.raise_for_status()

            with temp.open(
                "wb"
            ) as file:

                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):

                    if chunk:
                        file.write(chunk)

        temp.replace(destination)

    except Exception:

        try:
            temp.unlink(
                missing_ok=True
            )
        except Exception:
            pass

        raise

    return str(destination)


def inspect_torrent_file(torrent_path):
    import libtorrent as lt
    with open(torrent_path, "rb") as f:
        info = lt.torrent_info(lt.bdecode(f.read()))
    files = info.files()
    items = []
    for index in range(info.num_files()):
        try:
            path = files.file_path(index)
        except AttributeError:
            path = files.at(index).path
        items.append(str(path))
    return {"name": str(info.name()), "files": items}

def parse_file_selection(arg, file_count):
    value = (arg or "").strip().lower()
    if value == "all":
        return list(range(1, file_count + 1))
    selected = set()
    for token in value.split(","):
        token = token.strip()
        if not token: continue
        if "-" in token:
            a, b = [int(x.strip()) for x in token.split("-", 1)]
            if a > b: a, b = b, a
            selected.update(range(a, b + 1))
        else:
            selected.add(int(token))
    return sorted(i for i in selected if 1 <= i <= file_count)

def format_selected_names(files, selected):
    return "\n".join(f"{i}. {files[i-1]}" for i in selected if 1 <= i <= len(files))


def start_torrent_file(chat_id, file_id, filename):
    with active_lock:
        if active_task is not None:
            p = active_task.get("process")
            if p is None or p.poll() is None:
                send(chat_id, "⚠️ Another TeleDrive task is already running.")
                return
    with pending_selection_lock:
        if chat_id in pending_selections:
            send(chat_id, "⚠️ You already have a torrent waiting for selection. Use /select or /cancel.")
            return
    message_id = create_status_message(chat_id, "📥 Receiving .torrent file...")
    try:
        torrent_path = download_telegram_torrent(file_id, filename)
        metadata = inspect_torrent_file(torrent_path)
        if not metadata["files"]:
            raise RuntimeError("The torrent contains no files.")
        with pending_selection_lock:
            pending_selections[chat_id] = {"torrent_path": torrent_path, "message_id": message_id, "metadata": metadata, "created_at": time.time()}
        edit_message(chat_id, message_id, render_metadata(metadata))
    except Exception as exc:
        edit_message(chat_id, message_id, "❌ Could not receive .torrent:\n\n" + str(exc))

def start_selected_torrent(chat_id, selected):
    with pending_selection_lock:
        pending = pending_selections.pop(chat_id, None)
    if pending is None:
        send(chat_id, "ℹ️ No torrent is waiting for file selection. Send a .torrent first.")
        return
    selected = sorted(set(selected)); files = pending["metadata"]["files"]
    if not selected:
        with pending_selection_lock: pending_selections[chat_id] = pending
        send(chat_id, "⚠️ No valid file numbers were selected. Example: /select 1,2,3,...etc")
        return
    edit_message(chat_id, pending["message_id"], "🚀 Starting TeleDrive with selected files...\n\n" + f"📦 {pending['metadata']['name']}\n\n📥 Selected files:\n" + format_selected_names(files, selected))
    threading.Thread(target=run_cmd, args=(chat_id, [sys.executable, "main.py", "download", "-t", pending["torrent_path"], "--files", ",".join(map(str, selected)), "--upload"], pending["message_id"]), daemon=True).start()


# ------------------------------------------------------------
# Telegram command handler
# ------------------------------------------------------------

def handle_message(msg):
    chat_id = msg["chat"]["id"]

    uid = (
        msg.get("from", {})
        .get("id", "")
    )

    if not allowed(uid):

        send(
            chat_id,
            "⛔ You are not authorized to use this bot."
        )

        return

    # --------------------------------------------------------
    # Documents
    # --------------------------------------------------------

    document = msg.get("document")

    if document:

        filename = document.get(
            "file_name",
            "upload.torrent"
        )

        mime = (
            document.get("mime_type")
            or ""
        ).lower()

        if not (
            filename.lower().endswith(".torrent")
            or mime == "application/x-bittorrent"
        ):

            send(
                chat_id,
                "⚠️ Please send a .torrent file."
            )

            return

        threading.Thread(
            target=start_torrent_file,
            args=(
                chat_id,
                document["file_id"],
                filename,
            ),
            daemon=True
        ).start()

        return

    # --------------------------------------------------------
    # Text
    # --------------------------------------------------------

    text = (
        msg.get("text") or ""
    ).strip()

    if not text:
        return

    parts = text.split(
        maxsplit=1
    )

    command = (
        parts[0]
        .split("@")[0]
        .lower()
    )

    arg = (
        parts[1].strip()
        if len(parts) > 1
        else ""
    )

    # --------------------------------------------------------
    # /start
    # --------------------------------------------------------

    if command == "/start":

        send(
            chat_id,
            "🤖 TeleDrive Control\n\n"
            "📎 ابعت ملف .torrent مباشرة.\n"
            "🔗 /download <magnet link>\n"
            "📊 /status\n"
            "📤 /upload <local path>\n"
            "🧹 /clear\n"
            "🛑 /stop\n"
            "🔴 /shutdown\n"
            "🏓 /ping\n\n"
            "استخدم /help للمزيد."
        )

    # --------------------------------------------------------
    # /select
    # --------------------------------------------------------

    elif command == "/select":
        with pending_selection_lock:
            pending = pending_selections.get(chat_id)
        if pending is None:
            send(chat_id, "ℹ️ No torrent is waiting for selection. Send a .torrent first.")
            return
        try:
            selected = parse_file_selection(arg, len(pending["metadata"]["files"]))
        except ValueError:
            send(chat_id, "⚠️ Invalid selection. Use /select 1,2,3,...etc or /select 1,2,3,...etc or /select all.")
            return
        if not selected:
            send(chat_id, f"⚠️ No valid file numbers selected. Choose 1 to {len(pending['metadata']['files'])}.")
            return
        start_selected_torrent(chat_id, selected)

    # --------------------------------------------------------
    # /cancel
    # --------------------------------------------------------

    elif command == "/cancel":
        with pending_selection_lock:
            pending = pending_selections.pop(chat_id, None)
        if pending is None:
            send(chat_id, "ℹ️ No pending torrent selection.")
            return
        try:
            Path(pending["torrent_path"]).unlink(missing_ok=True)
        except Exception:
            pass
        edit_message(chat_id, pending["message_id"], "🛑 Torrent selection cancelled.")

    # --------------------------------------------------------
    # /help
    # --------------------------------------------------------

    elif command == "/help":

        send(
            chat_id,
            "🤖 TeleDrive Commands\n\n"
            "📎 Send a .torrent file\n"
            "→ The bot lists its files first\n"
            "→ /select 1,2,3,...etc downloads only those files\n"
            "→ /select all downloads everything\n"
            "→ /cancel cancels the pending selection\n\n"
            "🔗 /download <magnet link>\n"
            "→ تشغيل download في TeleDrive\n\n"
            "📤 /upload <local path>\n"
            "→ تشغيل upload في TeleDrive\n\n"
            "🔍 /status\n"
            "→ حالة المهمة\n\n"
            "🧹 /clear\n"
            "→ تنفيذ أمر clear في TeleDrive\n\n"
            "🛑 /stop\n"
            "→ إيقاف المهمة الحالية\n\n"
            "🏓 /ping\n"
            "→ اختبار البوت\n\n"
            "🔴 /shutdown\n"
            "→ إغلاق البوت"
        )

    # --------------------------------------------------------
    # /ping
    # --------------------------------------------------------

    elif command == "/ping":

        send(
            chat_id,
            "🏓 Bot is running."
        )

    # --------------------------------------------------------
    # /status
    # --------------------------------------------------------

    elif command == "/status":

        with active_lock:
            task = active_task

        if task is not None:

            process = task.get(
                "process"
            )

            if process is None:

                send(
                    chat_id,
                    "📊 TeleDrive task is starting..."
                )

            elif process.poll() is None:

                elapsed = int(
                    time.time()
                    - task["started_at"]
                )

                send(
                    chat_id,
                    "📊 TeleDrive is running.\n"
                    f"⏱ Running for {elapsed}s"
                )

            else:

                send(
                    chat_id,
                    "📊 TeleDrive process is finishing..."
                )

        else:

            # Keep compatibility with projects that provide
            # a main.py status command.
            try:

                result = subprocess.run(
                    [
                        sys.executable,
                        "main.py",
                        "status",
                    ],
                    capture_output=True,
                    text=True,
                    timeout=20
                )

                output = (
                    result.stdout.strip()
                    or result.stderr.strip()
                )

                send(
                    chat_id,
                    output
                    or "ℹ️ No active TeleDrive task."
                )

            except Exception as exc:

                send(
                    chat_id,
                    "❌ Status error:\n"
                    + str(exc)
                )

    # --------------------------------------------------------
    # /clear
    # --------------------------------------------------------

    elif command == "/clear":

        with active_lock:
            task = active_task

        if task is not None:

            process = task.get(
                "process"
            )

            if (
                process is None
                or process.poll() is None
            ):

                send(
                    chat_id,
                    "⚠️ Stop the active task before using /clear."
                )

                return

        message_id = create_status_message(
            chat_id,
            "🧹 Running TeleDrive clear..."
        )

        def clear_worker():

            try:

                result = subprocess.run(
                    [
                        sys.executable,
                        "main.py",
                        "clear",
                    ],
                    capture_output=True,
                    text=True,
                    timeout=120
                )

                output = (
                    result.stdout.strip()
                    or result.stderr.strip()
                    or "Clear command finished."
                )

                if result.returncode == 0:

                    edit_message(
                        chat_id,
                        message_id,
                        "✅ TeleDrive clear finished.\n\n"
                        + output
                    )

                else:

                    edit_message(
                        chat_id,
                        message_id,
                        "❌ TeleDrive clear failed.\n\n"
                        + output
                        + f"\n\nExit code: {result.returncode}"
                    )

            except Exception as exc:

                edit_message(
                    chat_id,
                    message_id,
                    "❌ Clear error:\n\n"
                    + str(exc)
                )

        threading.Thread(
            target=clear_worker,
            daemon=True
        ).start()

    # --------------------------------------------------------
    # /upload
    # --------------------------------------------------------

    elif command == "/upload":

        if not arg:

            send(
                chat_id,
                "⚠️ Usage:\n"
                "/upload <local file or folder path>"
            )

            return

        path = Path(
            arg
        ).expanduser().resolve()

        if not path.exists():

            send(
                chat_id,
                f"⚠️ Path does not exist:\n{path}"
            )

            return

        message_id = create_status_message(
            chat_id,
            "📤 Starting upload..."
        )

        threading.Thread(
            target=run_cmd,
            args=(
                chat_id,
                [
                    sys.executable,
                    "main.py",
                    "upload",
                    "-p",
                    str(path),
                ],
                message_id,
            ),
            daemon=True
        ).start()

    # --------------------------------------------------------
    # /download
    # --------------------------------------------------------

    elif command == "/download":

        if not arg:

            send(
                chat_id,
                "⚠️ Usage:\n"
                "/download <magnet link>"
            )

            return

        if not (
            arg.startswith(
                "magnet:"
            )
        ):

            send(
                chat_id,
                "⚠️ /download expects a magnet link."
            )

            return

        message_id = create_status_message(
            chat_id,
            "📥 Starting download..."
        )

        threading.Thread(
            target=run_cmd,
            args=(
                chat_id,
                [
                    sys.executable,
                    "main.py",
                    "download",
                    "-t",
                    arg,
                    "--upload",
                ],
                message_id,
            ),
            daemon=True
        ).start()

    # --------------------------------------------------------
    # /stop
    # --------------------------------------------------------

    elif command == "/stop":

        with active_lock:
            task = active_task

        if task is None:

            send(
                chat_id,
                "ℹ️ No active TeleDrive task."
            )

            return

        # Only the chat that started the task may stop it.
        if str(
            task.get("chat_id")
        ) != str(chat_id):

            send(
                chat_id,
                "⛔ You cannot stop another user's task."
            )

            return

        task["stop_event"].set()

        process = task.get(
            "process"
        )

        if process is not None:

            edit_message(
                task["chat_id"],
                task["message_id"],
                "🛑 Stopping TeleDrive..."
            )

            terminate_process(
                process
            )

        else:

            send(
                chat_id,
                "🛑 Stop requested; task is still starting."
            )

    # --------------------------------------------------------
    # /shutdown
    # --------------------------------------------------------

    elif command == "/shutdown":

        with active_lock:
            task = active_task

        if task is not None:

            task["stop_event"].set()

            process = task.get(
                "process"
            )

            if process is not None:

                edit_message(
                    task["chat_id"],
                    task["message_id"],
                    "🛑 Shutting down TeleDrive..."
                )

                terminate_process(
                    process
                )

        shutdown_event.set()

        send(
            chat_id,
            "🔴 TeleDrive bot is shutting down..."
        )

    # --------------------------------------------------------
    # Unknown
    # --------------------------------------------------------

    else:

        send(
            chat_id,
            "❓ Unknown command.\n"
            "Use /help."
        )


# ------------------------------------------------------------
# Long polling
# ------------------------------------------------------------

# IMPORTANT:
# Do not use:
#     while not shutdown_event.is_set():
#
# as the outer condition. If the Event was left set by a previous
# execution/state transition, the bot would print "running" and then
# immediately exit without ever calling getUpdates.
#
# The bot now runs continuously and only breaks explicitly when
# /shutdown or KeyboardInterrupt requests it.

# ------------------------------------------------------------
# Initialize Telegram polling from the newest update
# ------------------------------------------------------------
# Telegram keeps pending updates on the bot side. If the notebook
# is restarted after an old /shutdown command, processing that old
# update would immediately terminate the new session.
#
# We therefore discard updates that existed BEFORE this polling
# session starts. Commands sent AFTER startup are processed normally.

offset = None

print("🤖 Telegram bot is running.")
print("📎 Send a .torrent file to start a task.")
print("📊 Progress is updated in one Telegram message.")
print("🐍 Python:", sys.executable)

shutdown_event.clear()

try:
    startup_result = tg(
        "getUpdates",
        {
            "timeout": 0,
            "allowed_updates": ["message"],
        },
        timeout=10,
    )

    startup_updates = startup_result.get("result", [])

    if startup_updates:
        offset = startup_updates[-1]["update_id"] + 1
        print(
            f"🧹 Ignored {len(startup_updates)} old Telegram update(s)."
        )

except requests.RequestException as exc:
    print(
        "⚠️ Could not clear old Telegram updates:",
        repr(exc)
    )

except Exception as exc:
    print(
        "⚠️ Telegram startup warning:",
        repr(exc)
    )

print("🟢 Polling Telegram updates...")

while True:

    if shutdown_event.is_set():
        break

    try:

        data = {
            "timeout": 30,
        }

        if offset is not None:
            data["offset"] = offset

        result = tg(
            "getUpdates",
            data,
            timeout=40
        )

        # A successful getUpdates call proves that the bot is alive.
        # Keep the loop alive even when Telegram returns an empty list.
        updates = result.get("result", [])

        for update in updates:

            offset = update["update_id"] + 1

            message = update.get("message")

            if not message:
                continue

            try:

                handle_message(message)

            except Exception as exc:

                print(
                    "❌ Handler error:",
                    repr(exc)
                )

                try:

                    chat_id = message["chat"]["id"]

                    send(
                        chat_id,
                        "❌ Internal bot error:\n"
                        + str(exc)
                    )

                except Exception:
                    pass

    except KeyboardInterrupt:

        print("🛑 KeyboardInterrupt received.")
        shutdown_event.set()
        break

    except requests.exceptions.Timeout:

        # Normal for long polling when Telegram has no updates
        # for the requested timeout period.
        continue

    except requests.RequestException as exc:

        print(
            "⚠️ Telegram network error:",
            repr(exc)
        )

        # Do not terminate the bot because of a temporary network
        # failure. Wait briefly and reconnect.
        time.sleep(3)
        continue

    except Exception as exc:

        print(
            "⚠️ Polling error:",
            repr(exc)
        )

        # Keep the bot alive after unexpected polling errors.
        time.sleep(3)
        continue

print("🛑 Polling loop ended.")

# ------------------------------------------------------------
# Final cleanup
# ------------------------------------------------------------

with active_lock:
    task = active_task

if task is not None:

    task["stop_event"].set()

    process = task.get(
        "process"
    )

    if process is not None:
        terminate_process(
            process,
            force=False
        )

        try:
            process.wait(
                timeout=5
            )
        except Exception:
            terminate_process(
                process,
                force=True
            )

with active_lock:
    active_task = None

print("🔴 TeleDrive bot stopped.")


## Usage

1. Run cells 0 through 8 in order.
2. Leave the final bot cell running.
3. From Telegram:
   - 📎 Send a `.torrent` file directly.
   - 🔗 `/download <magnet link>`
   - 📤 `/upload <local path>`
   - 🔍 `/status`
   - 🧹 `/clear`
   - 🛑 `/stop`
   - 🔴 `/shutdown`

### Important Notes

- Never put your Telegram token directly in the code.
- Set `TELEGRAM_ALLOWED_USER_ID` so other users cannot control the bot.
- Only one TeleDrive task runs at a time.
- Each task has its own stop event.
- The controller reads real torrent statistics from libtorrent through `TD_PROGRESS`.
- Use only files and content that you have the right to download or upload.
